In [19]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize

## Load and Clean Corporate Bond Data

In [20]:
df = pd.read_excel("./data/cleaneddata.xlsx")
df.columns = df.columns.str.strip()

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values(["cusip", "date"])

for col in ["spread", "price", "sduration", "coupon", "ytm"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["date", "spread", "price", "sduration"])
df["spread"] = df["spread"] / 10000.0

df = df[df["date"].between("2017-01-01", "2019-12-31")]
df["ym"] = df["date"].dt.to_period("M")

## Build Monthly Panel and Compute Returns and DTS

In [21]:
price_col = "price"

# one observation per cusip per month (first available date)
first_day = (
    df.sort_values("date")
      .groupby(["cusip", "ym"])
      .first()
      .reset_index()
)

# next month price and next month label (within each cusip)
first_day["next_price"] = first_day.groupby("cusip")[price_col].shift(-1)
first_day["next_ym"]    = first_day.groupby("cusip")["ym"].shift(-1)

# drop incomplete rows
first_day = first_day.dropna(subset=["next_price", "next_ym"])

# enforce both current and next month in sample window
first_day = first_day[
    first_day["ym"].between("2017-01", "2019-12") &
    first_day["next_ym"].between("2017-01", "2019-12")
]

# price return
first_day["price_ret"] = (
    (first_day["next_price"] - first_day[price_col]) /
     first_day[price_col]
)

# carry = (annual coupon% / 12) / price
first_day["carry"] = ((first_day["coupon"] / 100) / 12) / first_day[price_col]

# total return
first_day["ret"] = first_day["price_ret"] + first_day["carry"]

# DTS = spread × spread duration
first_day["dts"] = first_day["spread"] * first_day["sduration"]

# wide return matrix (ym × cusip)
ret_pivot = (
    first_day.pivot(index="ym", columns="cusip", values="ret")
    .sort_index()
)

## Portfolio Optimizer (Duration Neutral, Long/Short)

In [22]:
def optimize_duration_neutral(mu, cov, sdur, w_max=0.10):
    n = len(mu)
    w0 = np.ones(n) / n

    def neg_sharpe(w):
        ret = np.dot(w, mu)
        vol = np.sqrt(np.dot(w.T, np.dot(cov, w)))
        if vol <= 0:
            return 1e9
        return -(ret / vol)

    bounds = [(-w_max, w_max)] * n

    cons = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {"type": "eq", "fun": lambda w: np.dot(w, sdur)}  # duration = 0
    ]

    res = minimize(
        neg_sharpe,
        w0,
        method="SLSQP",
        bounds=bounds,
        constraints=cons,
        options={"maxiter": 1000, "ftol": 1e-9}
    )

    if not res.success:
        return w0

    return res.x

## Build Unhedged Portfolios

In [23]:
dn_low  = []
dn_mid  = []
dn_high = []

months = sorted(first_day["ym"].unique())

for idx in range(3, len(months)):
    ym = months[idx]

    # trailing 3 months for mean/cov estimation
    window = months[idx-3:idx]
    hist = ret_pivot.loc[window]

    # current month bonds
    month_df = first_day[first_day["ym"] == ym].copy()
    month_df = month_df.sort_values("dts")
    n = len(month_df)
    if n < 6:
        continue

    # split into 3 equal DTS buckets
    k = n // 3
    low  = month_df.iloc[:k].copy()
    mid  = month_df.iloc[k:2*k].copy()
    high = month_df.iloc[2*k:3*k].copy()

    def prep(bucket):
        # align bucket cusips with historical return matrix
        cus = list(bucket["cusip"])
        sub = hist[cus].dropna(axis=1, how="any")
        if sub.shape[1] < 2:
            return None, None, None

        cusips = list(sub.columns)
        bucket2 = bucket.set_index("cusip").loc[cusips].reset_index()

        mu  = sub.mean().values
        cov = np.cov(sub.T)

        return bucket2, mu, cov

    low_b,  mu_low,  cov_low  = prep(low)
    mid_b,  mu_mid,  cov_mid  = prep(mid)
    high_b, mu_high, cov_high = prep(high)

    if low_b is None or mid_b is None or high_b is None:
        continue

    # spread-duration arrays
    sdur_low  = low_b["sduration"].values
    sdur_mid  = mid_b["sduration"].values
    sdur_high = high_b["sduration"].values

    # duration-neutral weights (using only corporates)
    w_low  = optimize_duration_neutral(mu_low,  cov_low,  sdur_low)
    w_mid  = optimize_duration_neutral(mu_mid,  cov_mid,  sdur_mid)
    w_high = optimize_duration_neutral(mu_high, cov_high, sdur_high)

    # realized corporate-only return (no external hedge yet)
    ret_low  = np.dot(w_low,  low_b["ret"].values)
    ret_mid  = np.dot(w_mid,  mid_b["ret"].values)
    ret_high = np.dot(w_high, high_b["ret"].values)

    # portfolio DTS exposure (for CDX hedge)
    dts_low  = np.dot(w_low,  low_b["dts"].values)
    dts_mid  = np.dot(w_mid,  mid_b["dts"].values)
    dts_high = np.dot(w_high, high_b["dts"].values)

    dn_low.append({
        "month": ym,
        "ret_dn": ret_low,
        "dts_p": dts_low
    })

    dn_mid.append({
        "month": ym,
        "ret_dn": ret_mid,
        "dts_p": dts_mid
    })

    dn_high.append({
        "month": ym,
        "ret_dn": ret_high,
        "dts_p": dts_high
    })

low_opt  = pd.DataFrame(dn_low)
mid_opt  = pd.DataFrame(dn_mid)
high_opt = pd.DataFrame(dn_high)

# cumulative returns for duration-neutral portfolios (no CDX hedge yet)
low_opt["cum_dn"]  = (1 + low_opt["ret_dn"]).cumprod() - 1
mid_opt["cum_dn"]  = (1 + mid_opt["ret_dn"]).cumprod() - 1
high_opt["cum_dn"] = (1 + high_opt["ret_dn"]).cumprod() - 1

print("LOW DTS — Duration-Neutral (Corporate Only):")
print(low_opt.tail(), "\n")

print("MID DTS — Duration-Neutral (Corporate Only):")
print(mid_opt.tail(), "\n")

print("HIGH DTS — Duration-Neutral (Corporate Only):")
print(high_opt.tail(), "\n")

c:\Users\lopes\anaconda3\envs\deeplearning\lib\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
c:\Users\lopes\anaconda3\envs\deeplearning\lib\site-packages\scipy\optimize\_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
c:\Users\lopes\anaconda3\envs\deeplearning\lib\site-packages\scipy\optimize\_slsqp_py.py:493: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])
c:\Users\lopes\anaconda3\envs\deeplearning\lib\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
c:\Users\lopes\anaconda3\envs\deeplearning\lib\site-packages\scipy\optimize\_slsqp_py.py:439: RuntimeWarning: Values in x were outside b

LOW DTS — Duration-Neutral (Corporate Only):
      month    ret_dn     dts_p    cum_dn
27  2019-07  0.011007  0.049800  0.033017
28  2019-08  0.008194  0.054126  0.041481
29  2019-09  0.000650  0.056684  0.042158
30  2019-10 -0.005737  0.071589  0.036179
31  2019-11 -0.001395  0.063833  0.034733 

MID DTS — Duration-Neutral (Corporate Only):
      month    ret_dn     dts_p    cum_dn
27  2019-07  0.020399  0.241942  0.044396
28  2019-08  0.026006  0.248690  0.071557
29  2019-09 -0.008953  0.243072  0.061963
30  2019-10 -0.002778  0.271015  0.059012
31  2019-11  0.009313  0.252288  0.068875 

HIGH DTS — Duration-Neutral (Corporate Only):
      month    ret_dn     dts_p    cum_dn
27  2019-07  0.024322  0.403877  0.078948
28  2019-08  0.002343  0.407787  0.081476
29  2019-09  0.006757  0.404534  0.088784
30  2019-10  0.001678  0.439507  0.090611
31  2019-11  0.003748  0.431430  0.094699 



c:\Users\lopes\anaconda3\envs\deeplearning\lib\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)
c:\Users\lopes\anaconda3\envs\deeplearning\lib\site-packages\scipy\optimize\_slsqp_py.py:439: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  g = append(wrapped_grad(x), 0.0)
c:\Users\lopes\anaconda3\envs\deeplearning\lib\site-packages\scipy\optimize\_slsqp_py.py:493: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  a_eq = vstack([con['jac'](x, *con['args'])


## Load and Process CDX Data

In [24]:
cdx = pd.read_csv("./data/cdx_all_us_cleaned.csv")

# basic cleaning
cdx["date"] = pd.to_datetime(cdx["date"], errors="coerce")
cdx["spread_bps"] = pd.to_numeric(cdx["spread_bps"], errors="coerce")

# filter to US_IG 5Y only (one index)
cdx = cdx[(cdx["index_name"] == "US_IG") & (cdx["tenor"] == "5Y")].copy()

# restrict to same sample window
cdx = cdx[cdx["date"].between("2017-01-01", "2019-12-31")]

# convert to monthly (take last spread of each month)
cdx["ym"] = cdx["date"].dt.to_period("M")
cdx_m = (
    cdx.sort_values("date")
        .groupby("ym")
        .tail(1)    # last obs in month
        .set_index("ym")
        .sort_index()
)

# convert spread to decimal (e.g., 81.25 bps => 0.008125)
cdx_m["spread"] = cdx_m["spread_bps"] / 10000.0

# assume a constant CDX index duration (rough approximation)
duration_cdx = 5.0  # you can refine this if you have better data

# approximate monthly CDX 'price return' via -Duration * ΔSpread
cdx_m["delta_spread"] = cdx_m["spread"].diff()
cdx_m["ret_cdx"] = -duration_cdx * cdx_m["delta_spread"]

# DTS of one unit notional of CDX each month
cdx_m["dts_cdx"] = cdx_m["spread"] * duration_cdx

# drop first month with NaN ret_cdx
cdx_m = cdx_m.dropna(subset=["ret_cdx", "dts_cdx"])

# keep only needed columns
cdx_m = cdx_m[["ret_cdx", "dts_cdx"]]

## Merge Portfolios with CDX and Compute Hedge

In [25]:
for df_opt in [low_opt, mid_opt, high_opt]:
    df_opt["ym"] = df_opt["month"]
    df_opt.set_index("ym", inplace=True)

# join CDX monthly returns & DTS
low_opt  = low_opt.join(cdx_m, how="inner")
mid_opt  = mid_opt.join(cdx_m, how="inner")
high_opt = high_opt.join(cdx_m, how="inner")

# hedge ratios based on DTS: theta_cdx_t = - DTS_portfolio_t / DTS_CDX_t
low_opt["theta_cdx"]  = - low_opt["dts_p"]  / low_opt["dts_cdx"]
mid_opt["theta_cdx"]  = - mid_opt["dts_p"]  / mid_opt["dts_cdx"]
high_opt["theta_cdx"] = - high_opt["dts_p"] / high_opt["dts_cdx"]

# hedged returns: duration-neutral corporate + CDX overlay (DTS-neutral)
low_opt["ret_dn_hedged"]  = low_opt["ret_dn"]  + low_opt["theta_cdx"] * low_opt["ret_cdx"]
mid_opt["ret_dn_hedged"]  = mid_opt["ret_dn"]  + mid_opt["theta_cdx"] * mid_opt["ret_cdx"]
high_opt["ret_dn_hedged"] = high_opt["ret_dn"] + high_opt["theta_cdx"] * high_opt["ret_cdx"]

# cumulative hedged returns
low_opt["cum_dn_hedged"]  = (1 + low_opt["ret_dn_hedged"]).cumprod() - 1
mid_opt["cum_dn_hedged"]  = (1 + mid_opt["ret_dn_hedged"]).cumprod() - 1
high_opt["cum_dn_hedged"] = (1 + high_opt["ret_dn_hedged"]).cumprod() - 1

print("LOW DTS — Duration-Neutral + DTS-Neutral (CDX Hedged):")
print(low_opt[["month", "ret_dn", "dts_p", "dts_cdx", "theta_cdx", "ret_dn_hedged", "cum_dn_hedged"]].tail(), "\n")
low_opt.to_csv('./portfolios/dts_hedged/low_opt.csv')

print("MID DTS — Duration-Neutral + DTS-Neutral (CDX Hedged):")
print(mid_opt[["month", "ret_dn", "dts_p", "dts_cdx", "theta_cdx", "ret_dn_hedged", "cum_dn_hedged"]].tail(), "\n")
mid_opt.to_csv('./portfolios/dts_hedged/mid_opt.csv')

print("HIGH DTS — Duration-Neutral + DTS-Neutral (CDX Hedged):")
print(high_opt[["month", "ret_dn", "dts_p", "dts_cdx", "theta_cdx", "ret_dn_hedged", "cum_dn_hedged"]].tail())
high_opt.to_csv('./portfolios/dts_hedged/high_opt.csv')

LOW DTS — Duration-Neutral + DTS-Neutral (CDX Hedged):
           month    ret_dn     dts_p   dts_cdx  theta_cdx  ret_dn_hedged  \
ym                                                                         
2019-07  2019-07  0.011007  0.049800  0.026075  -1.909888       0.007903   
2019-08  2019-08  0.008194  0.054126  0.026870  -2.014370       0.009795   
2019-09  2019-09  0.000650  0.056684  0.029900  -1.895789       0.006395   
2019-10  2019-10 -0.005737  0.071589  0.027528  -2.600573      -0.011906   
2019-11  2019-11 -0.001395  0.063833  0.025015  -2.551805      -0.007808   

         cum_dn_hedged  
ym                      
2019-07      -0.007228  
2019-08       0.002496  
2019-09       0.008907  
2019-10      -0.003105  
2019-11      -0.010889   

MID DTS — Duration-Neutral + DTS-Neutral (CDX Hedged):
           month    ret_dn     dts_p   dts_cdx  theta_cdx  ret_dn_hedged  \
ym                                                                         
2019-07  2019-07  0.020399  